# 02 — Behavioral Metrics
**UIDAI Aadhaar Fraud Intelligence System — Thane District**

---

### Objective
While Ghost Detection (Notebook 01) identifies **fraudulent centers**,
this notebook profiles **how each pincode behaves operationally** across 4 dimensions:

| Metric | Full Name | What it measures |
|---|---|---|
| **ICI** | Identity Churn Index | Adult identity instability |
| **BDR** | Biometric Distress Ratio | Biometric pressure vs demographic updates |
| **CBCR** | Child Biometric Compliance Ratio | Children completing mandatory biometric updates |
| **ES** | Enrollment Saturation | Is the area still enrolling or already saturated? |

### Output
A single **`pin_summary.csv`** — one row per pincode, all behavioral flags included.

### Pipeline
```
master_aadhaar_thane.csv
        ↓
Compute ICI → flag persistent churn pincodes
        ↓
Compute BDR → flag biometric distress zones
        ↓
Compute CBCR → flag low child compliance pincodes
        ↓
Compute Enrollment Saturation → flag saturated areas
        ↓
Merge all → pin_summary.csv → reports/
```

## Cell 1 — Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
import os

# ── Paths ─────────────────────────────────────────────────────
DATA_PATH   = '../data/processed/master_aadhaar_thane.csv'
REPORTS_DIR = '../reports'
os.makedirs(REPORTS_DIR, exist_ok=True)

# ── Load ──────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)

# ── Recompute derived columns needed for metrics ──────────────
df['enrol_total'] = df['enrol_infant'] + df['enrol_child'] + df['enrol_adult']
df['total_txn']   = df['bio_total'] + df['demo_total'] + df['enrol_total']

print(f"Rows     : {len(df)}")
print(f"Pincodes : {df['pincode'].nunique()}")
print(f"Months   : {sorted(df['month'].unique())}")
df.head()

Rows     : 982
Pincodes : 96
Months   : ['2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01']


,pincode,month,bio_child,bio_adult,bio_total,demo_child,demo_adult,demo_total,enrol_infant,enrol_child,enrol_adult,enrol_total,total_txn
0,400601,2025-03,0.0,0.0,0.0,99.0,1058.0,1157.0,0.0,0.0,0.0,0.0,1157.0
1,400601,2025-04,393.0,831.0,1224.0,104.0,783.0,887.0,0.0,0.0,0.0,0.0,2111.0
2,400601,2025-05,575.0,1122.0,1697.0,0.0,0.0,0.0,80.0,30.0,14.0,124.0,1821.0
3,400601,2025-06,165.0,670.0,835.0,0.0,0.0,0.0,45.0,31.0,1.0,77.0,912.0
4,400601,2025-07,407.0,912.0,1319.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1319.0


## Cell 2 — Metric 1: ICI (Identity Churn Index)

**Formula:** `ICI = (demo_adult + bio_adult) / (total_txn + 1)`

**Interpretation:**
- High ICI → a large fraction of all activity is adult identity updates (name, address, biometric)
- Persistently high ICI across months → structural instability, possible migration zone or fraud

**Flagging strategy:**
- We flag **per month** using the 90th percentile threshold
- We then aggregate to **per pincode** — counting how frequently each pincode hits the top 10%
- Frequency is more meaningful than a single snapshot

In [2]:
# ── Compute ICI per pincode-month ─────────────────────────────
df['ICI'] = (df['demo_adult'] + df['bio_adult']) / (df['total_txn'] + 1)

# ── Thresholds ────────────────────────────────────────────────
ici_p90 = df['ICI'].quantile(0.90)
ici_p75 = df['ICI'].quantile(0.75)

print(f"ICI 75th percentile : {ici_p75:.4f}")
print(f"ICI 90th percentile : {ici_p90:.4f}")
print(f"ICI mean            : {df['ICI'].mean():.4f}")

# ── Flag pin-months ───────────────────────────────────────────
df['ici_top10'] = df['ICI'] >= ici_p90
df['ici_top25'] = df['ICI'] >= ici_p75

print(f"\nPin-months in top 10% ICI : {df['ici_top10'].sum()}")
print(f"Pin-months in top 25% ICI : {df['ici_top25'].sum()}")

# ── Aggregate to pincode level ────────────────────────────────
ici_pin = (
    df.groupby('pincode')
      .agg(
          months_total  = ('month',    'nunique'),
          top10_count   = ('ici_top10','sum'),
          top25_count   = ('ici_top25','sum')
      )
      .reset_index()
)

# Frequency fraction (accounts for pincodes with different data lengths)
ici_pin['top10_frac'] = ici_pin['top10_count'] / ici_pin['months_total']
ici_pin['top25_frac'] = ici_pin['top25_count'] / ici_pin['months_total']

print("\nTop 10 persistently churning pincodes:")
display(
    ici_pin.sort_values('top10_frac', ascending=False)
           .head(10)[['pincode','months_total','top10_count','top10_frac']]
           .reset_index(drop=True)
)

total_pins   = len(ici_pin)
never_top10  = (ici_pin['top10_frac'] == 0).sum()
print(f"\nPincodes never in top-10% ICI : {never_top10} / {total_pins}")
print(f"Pincodes with persistent churn : {total_pins - never_top10} / {total_pins}")

ICI 75th percentile : 0.9091
ICI 90th percentile : 0.9565
ICI mean            : 0.7776

Pin-months in top 10% ICI : 99
Pin-months in top 25% ICI : 248

Top 10 persistently churning pincodes:


,pincode,months_total,top10_count,top10_frac
0,401103,9,6,0.666667
1,401605,9,6,0.666667
2,401702,9,6,0.666667
3,401403,9,6,0.666667
4,401603,9,6,0.666667
5,401503,9,5,0.555556
6,401405,9,5,0.555556
7,401607,10,5,0.500000
8,401501,11,5,0.454545
9,401601,9,4,0.444444



Pincodes never in top-10% ICI : 68 / 96
Pincodes with persistent churn : 28 / 96


## Cell 3 — Metric 2: BDR (Biometric Distress Ratio)

**Formula:** `BDR = bio_adult / (demo_adult + 1)`

**Interpretation:**
- BDR > 1 → more biometric updates than demographic updates (biometric pressure)
- BDR >> 1 → adults updating biometrics with **zero demographic activity** → distress signal
- `biometric_only` flag → months where `demo_adult == 0` entirely

**ICI × BDR Interpretation Matrix:**

| ICI | BDR | Zone |
|---|---|---|
| High | High | 🔴 Biometric Distress Zone |
| High | Low | 🟡 Migration / Address Churn |
| Low | High | 🟠 Isolated Biometric Problem |

In [3]:
# ── Compute BDR ───────────────────────────────────────────────
df['BDR'] = df['bio_adult'] / (df['demo_adult'] + 1)

# ── biometric_only flag ───────────────────────────────────────
# Months where demo_adult == 0 (no demographic activity at all)
df['biometric_only'] = df['demo_adult'] == 0

# ── BDR_valid: only compute where demo > 0 ────────────────────
# Avoids inflated ratios from division near zero
df['BDR_valid'] = np.where(
    df['demo_adult'] > 0,
    df['bio_adult'] / df['demo_adult'],
    np.nan
)

# ── Cap at 95th percentile to handle extreme outliers ─────────
bdr_cap = df['BDR_valid'].quantile(0.95)
df['BDR_capped'] = df['BDR_valid'].clip(upper=bdr_cap)

print(f"BDR 95th percentile cap : {bdr_cap:.4f}")
print(f"Months with biometric_only : {df['biometric_only'].sum()} / {len(df)}")

# ── Aggregate to pincode level ────────────────────────────────
bdr_pin = (
    df.groupby('pincode')
      .agg(
          months_total            = ('month',          'nunique'),
          biometric_only_count    = ('biometric_only', 'sum'),
          median_BDR              = ('BDR_capped',     'median'),
          mean_BDR                = ('BDR_capped',     'mean')
      )
      .reset_index()
)

bdr_pin['biometric_only_frac'] = bdr_pin['biometric_only_count'] / bdr_pin['months_total']

# ── Flag high-distress pincodes ───────────────────────────────
bdr_pin['high_BDR']         = bdr_pin['median_BDR']          >= bdr_pin['median_BDR'].quantile(0.75)
bdr_pin['high_bio_only']    = bdr_pin['biometric_only_frac']  >= bdr_pin['biometric_only_frac'].quantile(0.75)

danger_zone = (bdr_pin['high_BDR'] & bdr_pin['high_bio_only']).sum()
print(f"\nPincodes in danger zone (high BDR + high bio-only) : {danger_zone}")

print("\nTop 10 biometric distress pincodes:")
display(
    bdr_pin.sort_values('median_BDR', ascending=False)
           .head(10)[['pincode','biometric_only_frac','median_BDR','high_BDR','high_bio_only']]
           .reset_index(drop=True)
)

BDR 95th percentile cap : 3.4733
Months with biometric_only : 429 / 982

Pincodes in danger zone (high BDR + high bio-only) : 13

Top 10 biometric distress pincodes:


,pincode,biometric_only_frac,median_BDR,high_BDR,high_bio_only
0,401703,0.444444,3.473327,True,False
1,401302,0.444444,3.473327,True,False
2,401304,0.500000,3.363636,True,True
3,401103,0.444444,2.791045,True,False
4,401203,0.545455,2.699301,True,True
5,401602,0.400000,2.533797,True,False
6,400603,0.454545,2.453373,True,True
7,421304,0.444444,2.300000,True,False
8,421001,0.400000,2.201890,True,False
9,401102,0.500000,2.065934,True,True


## Cell 4 — Metric 3: CBCR (Child Biometric Compliance Ratio)

**Formula:** `CBCR = bio_child / (bio_child + demo_child + 1)`

**Interpretation:**
- CBCR → 1.0 : children completing biometric updates (compliant)
- CBCR → 0.0 : children only doing demographic updates, skipping biometrics (non-compliant)
- CBCR = 0.0 for all months → complete non-compliance in that pincode

In [4]:
# ── Compute CBCR ──────────────────────────────────────────────
df['CBCR'] = df['bio_child'] / (df['bio_child'] + df['demo_child'] + 1)

print("CBCR distribution:")
print(df['CBCR'].describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]).round(4))

# ── Aggregate to pincode level ────────────────────────────────
cbcr_pin = (
    df.groupby('pincode')
      .agg(
          months_total = ('month', 'nunique'),
          mean_CBCR    = ('CBCR',  'mean'),
          median_CBCR  = ('CBCR',  'median')
      )
      .reset_index()
)

# ── Flag low-compliance pincodes (bottom 25%) ─────────────────
cbcr_cutoff = cbcr_pin['mean_CBCR'].quantile(0.25)
cbcr_pin['low_CBCR'] = cbcr_pin['mean_CBCR'] <= cbcr_cutoff

print(f"\nLow CBCR threshold (25th pct) : {cbcr_cutoff:.4f}")
print(f"Pincodes with low compliance  : {cbcr_pin['low_CBCR'].sum()}")

print("\nBottom 10 — lowest child compliance:")
display(
    cbcr_pin.sort_values('mean_CBCR')
            .head(10)[['pincode','months_total','mean_CBCR','median_CBCR','low_CBCR']]
            .reset_index(drop=True)
)

CBCR distribution:
count    982.0000
mean       0.8016
std        0.2799
min        0.0000
10%        0.5000
25%        0.7975
50%        0.9091
75%        0.9714
90%        0.9963
max        0.9995
Name: CBCR, dtype: float64

Low CBCR threshold (25th pct) : 0.7894
Pincodes with low compliance  : 24

Bottom 10 — lowest child compliance:


,pincode,months_total,mean_CBCR,median_CBCR,low_CBCR
0,401609,6,0.000000,0.0,True
1,401608,7,0.000000,0.0,True
2,421304,9,0.000000,0.0,True
3,421604,9,0.175926,0.0,True
4,401702,9,0.194444,0.0,True
5,401703,9,0.231481,0.0,True
6,401701,9,0.259259,0.0,True
7,401401,9,0.375926,0.5,True
8,401405,9,0.418519,0.5,True
9,401503,9,0.469481,0.5,True


## Cell 5 — Metric 4: Enrollment Saturation (ES)

**Formula:** `ES = enrol_total / (total_txn + 1)`

**Interpretation:**
- ES → 0 : area is **saturated** — almost no new Aadhaar creation, only updates
- ES → 1 : area is **actively enrolling** — large share of activity is new Aadhaar numbers

This metric helps UIDAI with **resource and capacity planning**.

In [5]:
# ── Compute Enrollment Saturation ────────────────────────────
df['enrol_sat'] = df['enrol_total'] / (df['total_txn'] + 1)

print("Enrollment Saturation distribution:")
print(df['enrol_sat'].describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]).round(4))

# ── Aggregate to pincode level ────────────────────────────────
enrol_pin = (
    df.groupby('pincode')
      .agg(
          months_total       = ('month',     'nunique'),
          mean_enrol_sat     = ('enrol_sat', 'mean'),
          median_enrol_sat   = ('enrol_sat', 'median'),
          mean_new_enrol     = ('enrol_total','mean')
      )
      .reset_index()
)

# ── Flag saturated pincodes (bottom 25%) ──────────────────────
sat_cutoff = enrol_pin['median_enrol_sat'].quantile(0.25)
enrol_pin['is_saturated'] = enrol_pin['median_enrol_sat'] <= sat_cutoff

print(f"\nSaturation threshold (25th pct) : {sat_cutoff:.6f}")
print(f"Saturated pincodes              : {enrol_pin['is_saturated'].sum()} / {len(enrol_pin)}")

print("\nTop 10 — still actively enrolling:")
display(
    enrol_pin.sort_values('median_enrol_sat', ascending=False)
             .head(10)[['pincode','months_total','median_enrol_sat','mean_new_enrol','is_saturated']]
             .reset_index(drop=True)
)

Enrollment Saturation distribution:
count    982.0000
mean       0.0272
std        0.0404
min        0.0000
10%        0.0000
25%        0.0000
50%        0.0112
75%        0.0417
90%        0.0716
max        0.3501
Name: enrol_sat, dtype: float64

Saturation threshold (25th pct) : 0.006932
Saturated pincodes              : 24 / 96

Top 10 — still actively enrolling:


,pincode,months_total,median_enrol_sat,mean_new_enrol,is_saturated
0,401107,11,0.058785,374.363636,False
1,421302,11,0.054461,765.454545,False
2,421503,11,0.044477,274.909091,False
3,400612,11,0.044096,541.818182,False
4,400601,11,0.043159,68.181818,False
5,421306,11,0.040207,210.636364,False
6,421203,11,0.038793,21.636364,False
7,421501,11,0.038654,138.363636,False
8,400607,11,0.036345,60.818182,False
9,400614,11,0.034722,14.272727,False


## Cell 6 — Build & Export pin_summary

Merge all 4 metric profiles into one comprehensive per-pincode summary.

In [6]:
# ── Merge all metric tables ───────────────────────────────────
pin_summary = ici_pin.copy()

pin_summary = pin_summary.merge(
    bdr_pin[['pincode','biometric_only_count','biometric_only_frac',
             'median_BDR','mean_BDR','high_BDR','high_bio_only']],
    on='pincode', how='left'
)

pin_summary = pin_summary.merge(
    cbcr_pin[['pincode','mean_CBCR','median_CBCR','low_CBCR']],
    on='pincode', how='left'
)

pin_summary = pin_summary.merge(
    enrol_pin[['pincode','mean_enrol_sat','median_enrol_sat','is_saturated']],
    on='pincode', how='left'
)

print(f"pin_summary shape : {pin_summary.shape}")
print(f"Columns           : {pin_summary.columns.tolist()}")

# ── Summary of flags ──────────────────────────────────────────
print("\n── Flag Summary ──────────────────────────")
print(f"  Persistent ICI churn (top10_frac > 0) : {(pin_summary['top10_frac'] > 0).sum()} pincodes")
print(f"  High biometric distress                : {pin_summary['high_BDR'].sum()} pincodes")
print(f"  Low child compliance                   : {pin_summary['low_CBCR'].sum()} pincodes")
print(f"  Enrollment saturated                   : {pin_summary['is_saturated'].sum()} pincodes")

display(pin_summary.head())

# ── Export ────────────────────────────────────────────────────
pin_summary.to_csv(f'{REPORTS_DIR}/pin_summary.csv', index=False)
print("\nSaved → reports/pin_summary.csv")

pin_summary shape : (96, 18)
Columns           : ['pincode', 'months_total', 'top10_count', 'top25_count', 'top10_frac', 'top25_frac', 'biometric_only_count', 'biometric_only_frac', 'median_BDR', 'mean_BDR', 'high_BDR', 'high_bio_only', 'mean_CBCR', 'median_CBCR', 'low_CBCR', 'mean_enrol_sat', 'median_enrol_sat', 'is_saturated']

── Flag Summary ──────────────────────────
  Persistent ICI churn (top10_frac > 0) : 28 pincodes
  High biometric distress                : 24 pincodes
  Low child compliance                   : 24 pincodes
  Enrollment saturated                   : 24 pincodes


,pincode,months_total,top10_count,top25_count,top10_frac,top25_frac,biometric_only_count,biometric_only_frac,median_BDR,mean_BDR,high_BDR,high_bio_only,mean_CBCR,median_CBCR,low_CBCR,mean_enrol_sat,median_enrol_sat,is_saturated
0,400601,11,0,1,0.0,0.090909,4,0.363636,1.264626,1.206950,False,False,0.849673,0.921008,False,0.038606,0.043159,False
1,400602,11,0,1,0.0,0.090909,5,0.454545,1.283665,1.077751,False,True,0.871655,0.940559,False,0.020339,0.022079,False
2,400603,11,0,0,0.0,0.000000,5,0.454545,2.453373,2.436514,True,True,0.963210,0.971129,False,0.016397,0.017782,False
3,400604,11,0,0,0.0,0.000000,5,0.454545,1.677916,1.640738,True,True,0.931830,0.951420,False,0.034150,0.019126,False
4,400605,11,0,0,0.0,0.000000,4,0.363636,1.321130,1.310749,False,False,0.907109,0.917843,False,0.029067,0.022066,False



Saved → reports/pin_summary.csv
